# NumPy Mini Project: Credit Card Portfolio Risk Report

You've covered fundamentals, advanced concepts, and tips/tricks separately. This notebook strings them together into one realistic pipeline, pure numpy, no pandas yet: clean a messy synthetic portfolio, engineer risk features, score and segment customers, then run a Monte Carlo simulation to estimate expected loss and Value-at-Risk.

**Format:** same as before — `YOUR CODE HERE` cell, then a `Solution` cell. Unlike the earlier notebooks, tasks here **build on each other in order** — this is a pipeline, not independent questions, so run cells top to bottom even if you skip attempting one.

Every solution below was run end-to-end against this exact dataset before being handed to you.

## Setup: the synthetic portfolio

Run this as-is — it generates 1000 synthetic customers with realistic messiness (missing values, a few over-limit balances). This is your dataset for the whole notebook.

In [ ]:
import numpy as np

rng = np.random.default_rng(2024)
n = 1000

customer_id = np.arange(100000, 100000 + n)
age = rng.integers(21, 75, size=n)
credit_score = rng.normal(680, 80, size=n).clip(300, 850).round().astype(float)
income = rng.normal(55000, 22000, size=n).clip(15000, None).round(2)
credit_limit = (income * rng.uniform(0.05, 0.25, size=n)).round(-2).clip(500, None)
current_balance = (credit_limit * rng.uniform(0, 1.1, size=n)).round(2)
num_late_payments = rng.poisson(1.2, size=n)
monthly_transactions = rng.poisson(18, size=n)

# inject some realistic missingness
missing_income_idx = rng.choice(n, size=int(0.03 * n), replace=False)
income[missing_income_idx] = np.nan
missing_score_idx = rng.choice(n, size=int(0.02 * n), replace=False)
credit_score[missing_score_idx] = np.nan

print(f"{n} customers generated")

### Task 1 — Inspect the raw data

For each of `age`, `credit_score`, `income`, `credit_limit`, `current_balance`: print its shape, dtype, and (for the float arrays) how many `NaN` values it has using `np.isnan(...).sum()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
for name, arr in [("age", age), ("credit_score", credit_score), ("income", income),
                   ("credit_limit", credit_limit), ("current_balance", current_balance)]:
    nan_count = np.isnan(arr).sum() if arr.dtype.kind == "f" else 0
    print(name, arr.shape, arr.dtype, "nan count:", nan_count)

### Task 2 — Clean missing values

Create `income_clean` and `credit_score_clean`: same as the originals, but every `NaN` replaced with that column's median (`np.nanmedian` gives you the median while ignoring NaNs — use `np.where` to do the replacement). Confirm zero NaNs remain in each.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
income_clean = np.where(np.isnan(income), np.nanmedian(income), income)
credit_score_clean = np.where(np.isnan(credit_score), np.nanmedian(credit_score), credit_score)
print(np.isnan(income_clean).sum(), np.isnan(credit_score_clean).sum())

### Task 3 — Compute credit utilization

Compute `credit_utilization` = `current_balance / credit_limit` for every customer (vectorized, no loop). Note some values may exceed 1.0 — that's a real phenomenon (over-limit accounts), not a bug.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
credit_utilization = current_balance / credit_limit
print(credit_utilization[:5])

### Task 4 — Flag high utilization accounts

Create a boolean array `high_utilization`, `True` where utilization exceeds 0.7. Print how many customers that flags.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
high_utilization = credit_utilization > 0.7
print(high_utilization.sum(), "of", n)

### Task 5 — Combined risk flag

Create `double_flagged`: `True` for customers who are **both** high-utilization **and** have 2 or more late payments (`num_late_payments >= 2`). Remember: `&` not `and`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
double_flagged = high_utilization & (num_late_payments >= 2)
print(double_flagged.sum())

### Task 6 — Risk tier classification

Using `credit_score_clean`, build `risk_tier` with `np.select`: `"Low"` if score >= 740, `"Medium"` if score >= 670, `"High"` if score >= 580, else `"Very High"`. Then print the unique tiers and their counts with `np.unique(..., return_counts=True)`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
conditions = [
    credit_score_clean >= 740,
    credit_score_clean >= 670,
    credit_score_clean >= 580,
]
choices = ["Low", "Medium", "High"]
risk_tier = np.select(conditions, choices, default="Very High")
unique_tiers, tier_counts = np.unique(risk_tier, return_counts=True)
print(unique_tiers, tier_counts)

### Task 7 — Standardize features into a matrix

Stack `credit_score_clean`, `credit_utilization`, `num_late_payments`, and `income_clean` into a single `feature_matrix` (use `np.column_stack`). Then z-score standardize each column — `(feature_matrix - column_means) / column_stds` — into `z_features`, using `axis=0` for the means and stds.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
feature_matrix = np.column_stack([credit_score_clean, credit_utilization, num_late_payments, income_clean])
feature_means = feature_matrix.mean(axis=0)
feature_stds = feature_matrix.std(axis=0)
z_features = (feature_matrix - feature_means) / feature_stds
print(z_features[:3].round(3))

### Task 8 — Weighted risk score

Given `weights = np.array([-0.4, 0.3, 0.2, -0.1])` (matching the 4 columns in `z_features`: higher score = lower risk, higher utilization/late payments = higher risk, higher income = lower risk), compute `weighted_risk_score` for every customer using matrix multiplication (`z_features @ weights`).

In [ ]:
weights = np.array([-0.4, 0.3, 0.2, -0.1])

# YOUR CODE HERE


**Solution**

In [ ]:
weights = np.array([-0.4, 0.3, 0.2, -0.1])
weighted_risk_score = z_features @ weights
print(weighted_risk_score[:5].round(3))

### Task 9 — Top 10 riskiest customers

Use `np.argsort` on `weighted_risk_score` to find the 10 highest-risk customers, and use those indices to pull their IDs from `customer_id` into `top_10_riskiest_ids` (highest risk first).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
top_10_idx = np.argsort(weighted_risk_score)[-10:][::-1]
top_10_riskiest_ids = customer_id[top_10_idx]
print(top_10_riskiest_ids)

### Task 10 — Correlation matrix

Compute `corr_matrix`, the correlation matrix between the 4 raw features in `feature_matrix` (credit score, utilization, late payments, income) using `np.corrcoef(..., rowvar=False)`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
corr_matrix = np.corrcoef(feature_matrix, rowvar=False)
print(corr_matrix.round(2))

### Task 11 — Risk by age segment

Using `age_bins = np.array([21, 30, 40, 50, 60, 75])`, bucket every customer's age with `np.digitize` into `age_bucket`. Then, looping over `np.unique(age_bucket)`, print each bucket's customer count and average `weighted_risk_score` (using a boolean mask per bucket — there's no `groupby` in pure numpy, this is exactly the kind of thing pandas will make one line later).

In [ ]:
age_bins = np.array([21, 30, 40, 50, 60, 75])

# YOUR CODE HERE


**Solution**

In [ ]:
age_bins = np.array([21, 30, 40, 50, 60, 75])
age_bucket = np.digitize(age, age_bins)
for bucket in np.unique(age_bucket):
    mask = age_bucket == bucket
    print("bucket", bucket, "n=", mask.sum(), "avg risk=", weighted_risk_score[mask].mean().round(3))

### Task 12 — Exposure-weighted average utilization

A simple average utilization treats a $200 balance the same as a $20,000 balance. Compute `exposure_weighted_utilization` instead, using `np.average` with `weights=current_balance` — customers carrying more debt count more toward the portfolio-level number.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
exposure_weighted_utilization = np.average(credit_utilization, weights=current_balance)
print(exposure_weighted_utilization.round(4))

### Task 13 — Map risk tiers to default probabilities

Given `tier_to_prob = {"Low": 0.01, "Medium": 0.05, "High": 0.15, "Very High": 0.35}`, build `default_prob`, a numeric array the same length as `risk_tier` where each customer gets the probability matching their tier. Use `np.select` (one condition per tier, comparing `risk_tier` to each tier name).

In [ ]:
tier_to_prob = {"Low": 0.01, "Medium": 0.05, "High": 0.15, "Very High": 0.35}

# YOUR CODE HERE


**Solution**

In [ ]:
tier_to_prob = {"Low": 0.01, "Medium": 0.05, "High": 0.15, "Very High": 0.35}
default_prob = np.select(
    [risk_tier == "Low", risk_tier == "Medium", risk_tier == "High", risk_tier == "Very High"],
    [tier_to_prob["Low"], tier_to_prob["Medium"], tier_to_prob["High"], tier_to_prob["Very High"]],
)
print(default_prob[:10])

### Task 14 — Monte Carlo simulation

Simulate 5000 possible portfolio outcomes at once (no loop over trials). Create `sim_rng = np.random.default_rng(99)`, then `random_draws = sim_rng.random((5000, n))`. A customer "defaults" in a trial if their random draw is below their `default_prob`. Assuming a loss-given-default of 60% (`loss_given_default = 0.6`), compute each trial's total `portfolio_losses` (sum of `balance * 0.6` across all customers who defaulted in that trial) — this should be a length-5000 array, one total loss per simulated scenario.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
trials = 5000
sim_rng = np.random.default_rng(99)
random_draws = sim_rng.random((trials, n))
defaults = random_draws < default_prob[None, :]
loss_given_default = 0.6
losses = defaults * current_balance[None, :] * loss_given_default
portfolio_losses = losses.sum(axis=1)
print(portfolio_losses[:5].round(2))

### Task 15 — Expected loss & 95% VaR

From `portfolio_losses`, compute `expected_loss` (the mean) and `var_95` (the 95th percentile — "in the worst 5% of scenarios, losses exceed this amount").

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
expected_loss = portfolio_losses.mean()
var_95 = np.percentile(portfolio_losses, 95)
print("expected_loss=", expected_loss.round(2), "var_95=", var_95.round(2))

### Task 16 — Save the cleaned dataset

Save `customer_id`, `age`, `credit_score_clean`, `income_clean`, `credit_limit`, `current_balance`, `num_late_payments`, `credit_utilization`, `risk_tier`, and `weighted_risk_score` into one compressed file at `/tmp/credit_risk_cleaned.npz` using `np.savez_compressed`, then reload it with `np.load` and print its keys to confirm.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
np.savez_compressed(
    "/tmp/credit_risk_cleaned.npz",
    customer_id=customer_id,
    age=age,
    credit_score=credit_score_clean,
    income=income_clean,
    credit_limit=credit_limit,
    current_balance=current_balance,
    num_late_payments=num_late_payments,
    credit_utilization=credit_utilization,
    risk_tier=risk_tier,
    weighted_risk_score=weighted_risk_score,
)
loaded = np.load("/tmp/credit_risk_cleaned.npz")
print(list(loaded.keys()))

### Task 17 — Final portfolio summary report

Print a short summary: total customers, average credit score, average utilization (as a percentage), count of high-utilization accounts, total exposure (sum of `current_balance`), expected loss, and 95% VaR.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(f"Customers: {n}")
print(f"Avg credit score: {credit_score_clean.mean():.1f}")
print(f"Avg utilization: {credit_utilization.mean():.2%}")
print(f"High utilization accounts: {high_utilization.sum()}")
print(f"Total exposure: {current_balance.sum():,.2f}")
print(f"Expected portfolio loss: {expected_loss:,.2f}")
print(f"95% VaR: {var_95:,.2f}")

## ✅ Checkpoint

**What you built:** a full, if small, credit risk pipeline in pure numpy — clean → engineer features → standardize → score → segment → simulate → report. Every technique here (masking, `np.select`, broadcasting, matrix multiplication, Monte Carlo via vectorized random draws, `np.percentile` for VaR) came from the fundamentals, advanced, and tips notebooks — this was about combining them under realistic conditions (missing data, correlated features, no convenient `groupby`).

**The rough edges you probably felt:** manually looping over age buckets in Task 11, and rebuilding boolean masks by hand for every new filter, are exactly the pain points pandas exists to remove.

**What's next:** pandas — Series, DataFrames, `.loc`/`.iloc`, filtering, `groupby`, merging — applied to this same kind of credit card transactions data, but far less friction.

Let me know when you're ready to move on, or if you want a second numpy mini-project first.